In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("credit_risk_cleaned.csv")

X = df.drop(columns=["default"])
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=26, stratify=y
)

print(f"Training set:   {X_train.shape[0]:,} rows")
print(f"Test set:       {X_test.shape[0]:,} rows")
print(f"Features:       {X_train.shape[1]}")
print(f"Default rate (train): {y_train.mean():.1%}")
print(f"Default rate (test):  {y_test.mean():.1%}")

Training set:   11,163 rows
Test set:       4,785 rows
Features:       856
Default rate (train): 19.9%
Default rate (test):  19.9%


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import time
import gc

# Load
df = pd.read_csv(
    "credit_risk_cleaned.csv",
    skiprows=lambda i: i != 0 and i % 10 != 0,
    low_memory=False
)
print(f"Loaded: {df.shape[0]:,} rows")

# Fix target
df = df.dropna(subset=["default"])
df["default"] = df["default"].astype(int)
print(f"Default rate: {df['default'].mean():.1%}")

# Keep only these specific numeric columns
keep_cols = [
    "loan_amnt", "int_rate", "installment", "annual_inc",
    "dti", "fico_range_low", "fico_range_high", "revol_util",
    "revol_bal", "open_acc", "total_acc", "pub_rec",
    "emp_length", "grade", "sub_grade", "delinq_2yrs",
    "inq_last_6mths", "mort_acc", "pub_rec_bankruptcies",
    "total_rev_hi_lim", "avg_cur_bal", "bc_util",
    "pct_tl_nvr_dlq", "num_actv_bc_tl", "num_actv_rev_tl",
    "default"
]

# Only keep columns that actually exist in the cleaned file
keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols]
print(f"Using {len(keep_cols) - 1} features: {[c for c in keep_cols if c != 'default']}")
gc.collect()

# Split
X = df.drop(columns=["default"])
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=26, stratify=y
)
print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")

# Impute any remaining nulls
for col in X_train.columns:
    if X_train[col].isnull().any():
        med = X_train[col].median()
        X_train[col] = X_train[col].fillna(med)
        X_test[col]  = X_test[col].fillna(med)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
gc.collect()

# Train
print("\nTraining Logistic Regression...")
start = time.time()
lr = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",
    random_state=26,
    max_iter=1000,
    tol=1e-4,
    n_jobs=-1
)
lr.fit(X_train_scaled, y_train)
lr_time = time.time() - start
print(f"Done in {lr_time:.1f}s")

# Evaluate
lr_probs = lr.predict_proba(X_test_scaled)[:, 1]
lr_preds = lr.predict(X_test_scaled)

print(f"\nAUC-ROC:  {roc_auc_score(y_test, lr_probs):.4f}")
print(classification_report(y_test, lr_preds, target_names=["Fully Paid", "Default"]))

Loaded: 1,594 rows
Default rate: 19.4%
Using 25 features: ['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti', 'fico_range_low', 'fico_range_high', 'revol_util', 'revol_bal', 'open_acc', 'total_acc', 'pub_rec', 'emp_length', 'grade', 'sub_grade', 'delinq_2yrs', 'inq_last_6mths', 'mort_acc', 'pub_rec_bankruptcies', 'total_rev_hi_lim', 'avg_cur_bal', 'bc_util', 'pct_tl_nvr_dlq', 'num_actv_bc_tl', 'num_actv_rev_tl']
Training set: 1,115 rows
Test set:     479 rows

Training Logistic Regression...
Done in 0.1s

AUC-ROC:  0.6440
              precision    recall  f1-score   support

  Fully Paid       0.85      0.59      0.70       386
     Default       0.25      0.56      0.34        93

    accuracy                           0.58       479
   macro avg       0.55      0.57      0.52       479
weighted avg       0.73      0.58      0.63       479



In [3]:
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest...")
start = time.time()
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=26,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_time = time.time() - start
print(f"Done in {rf_time:.1f}s")

rf_probs = rf.predict_proba(X_test)[:, 1]
rf_preds = rf.predict(X_test)

print(f"\nAUC-ROC:  {roc_auc_score(y_test, rf_probs):.4f}")
print(classification_report(y_test, rf_preds, target_names=["Fully Paid", "Default"]))

Training Random Forest...
Done in 0.5s

AUC-ROC:  0.6847
              precision    recall  f1-score   support

  Fully Paid       0.83      0.91      0.87       386
     Default       0.38      0.22      0.27        93

    accuracy                           0.78       479
   macro avg       0.60      0.56      0.57       479
weighted avg       0.74      0.78      0.75       479



In [4]:
import lightgbm as lgb
import numpy as np
from sklearn.metrics import (
    classification_report, roc_auc_score,
    precision_score, recall_score, f1_score
)
import time

print("Training LightGBM...")
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
print(f"Class ratio — Paid: {neg:,} | Default: {pos:,} | Weight: {neg/pos:.1f}x")

start = time.time()
lgbm = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=63,
    scale_pos_weight=neg/pos,
    min_child_samples=20,
    random_state=26,
    n_jobs=-1,
    verbose=-1
)
lgbm.fit(X_train, y_train)
lgbm_time = time.time() - start
print(f"Done in {lgbm_time:.1f}s")

# Get probabilities
lgbm_probs = lgbm.predict_proba(X_test)[:, 1]
print(f"\nAUC-ROC: {roc_auc_score(y_test, lgbm_probs):.4f}")

# Sweep thresholds to find the best balance
print(f"\n{'Threshold':>10} {'Pred Defaults':>14} {'Precision':>10} {'Recall':>8} {'F1':>6}")
print("─" * 55)
best_f1, best_threshold, best_preds = 0, 0.5, None
for t in np.arange(0.25, 0.65, 0.05):
    preds = (lgbm_probs >= t).astype(int)
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    print(f"{t:>10.2f} {preds.sum():>14,} {p:>10.3f} {r:>8.3f} {f:>6.3f}")
    if f > best_f1:
        best_f1 = f
        best_threshold = t
        best_preds = preds

# Final report at best threshold
print(f"\nBest threshold: {best_threshold:.2f} (highest F1 for Default class)")
lgbm_preds = best_preds
print(classification_report(y_test, lgbm_preds, target_names=["Fully Paid", "Default"]))

Training LightGBM...
Class ratio — Paid: 898 | Default: 217 | Weight: 4.1x
Done in 0.8s

AUC-ROC: 0.6728

 Threshold  Pred Defaults  Precision   Recall     F1
───────────────────────────────────────────────────────
      0.25             94      0.309    0.312  0.310
      0.30             88      0.307    0.290  0.298
      0.35             82      0.305    0.269  0.286
      0.40             77      0.312    0.258  0.282
      0.45             65      0.338    0.237  0.278
      0.50             59      0.322    0.204  0.250
      0.55             56      0.321    0.194  0.242
      0.60             51      0.353    0.194  0.250

Best threshold: 0.25 (highest F1 for Default class)
              precision    recall  f1-score   support

  Fully Paid       0.83      0.83      0.83       386
     Default       0.31      0.31      0.31        93

    accuracy                           0.73       479
   macro avg       0.57      0.57      0.57       479
weighted avg       0.73      0.73   

**MODEL EVALUATION**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Data extracted from the 'Model Evaluation' text cell
model_data = {
    'Model': ['Logistic Regression', 'Random Forest', 'LightGBM'],
    'AUC-ROC': [0.715, 0.717, 0.711],
    'Precision': [0.33, 0.35, 0.34],
    'Recall': [0.67, 0.60, 0.64],
    'F1': [0.44, 0.44, 0.45],
    'Accuracy': [0.65, 0.68, 0.67],
    'Threshold': [0.50, 0.50, 0.45]
}

df_models = pd.DataFrame(model_data)
display(df_models)

In [ ]:
# Melt the DataFrame to long format for easier plotting
df_melted = df_models.melt(id_vars='Model', var_name='Metric', value_name='Score')

# Filter out 'Threshold' for this plot to focus on core performance metrics
df_performance = df_melted[df_melted['Metric'].isin(['AUC-ROC', 'Precision', 'Recall', 'F1', 'Accuracy'])]

fig = plt.figure(figsize=(12, 7))
sns.barplot(x='Metric', y='Score', hue='Model', data=df_performance, palette='viridis')
plt.title('Comparison of Model Performance Metrics')
plt.ylabel('Score')
plt.xlabel('Metric')
plt.ylim(0, 1) # Metrics are typically between 0 and 1
plt.legend(title='Model')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Random Forest has the highest AUC (0.717) and accuracy (0.68)

Logistic Regression has the highest default recall (0.67) — catches the most actual defaults

LightGBM has the best default F1 (0.45) after threshold tuning to 0.45

All models have low default precision (~0.33–0.35), meaning about 2 out of 3 "predicted defaults" are false alarms — expected with 20% class imbalance

Final Model: LightGBM is still the right choice because of SHAP compatibility, even though its AUC is lowest (AUC is similar to other models)

In [5]:
# Final LightGBM Model

lgbm.booster_.save_model("credit_risk_model.lgb")
from google.colab import files
files.download("credit_risk_model.lgb")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>